# Try Saysafe in a notebook

From the repository root, open it with:

```bash
uv run --with jupyterlab --with ipykernel jupyter lab examples/saysafe_playground.ipynb
```

Run the cells from top to bottom to see two library checks: private reply routing and approval tied to an exact action. This uses fictional data, a simulated voice score, and no microphone, phone service, model API, or real tool action.

This demonstrates the library logic. A real app still has to connect these decisions to its speaker, private device, and action executor.

In [ ]:
import secrets
from decimal import Decimal

from saysafe import Action, ApprovalGuard, PrivacyGuard, Refused, Room

print('Saysafe is ready for a local demo.')

## 1. Keep a private reply off the speaker

The app reports that other people are nearby. Saysafe returns a short spoken line and the sensitive text for the private device.

In [ ]:
privacy = PrivacyGuard()
decision = privacy.check(
    'Your Chase verification code is 482913.',
    sources={'otp'},
    room=Room.OTHERS_PRESENT,
)

print('Speaker:', decision.say)
print('Private device:', decision.send_to_phone)

Expected: the speaker gets `I sent it to your phone.` and the private device gets the full code. Nothing is actually sent to a device in this notebook.

## 2. Ask for approval before a transfer

This creates a fake $25 transfer. The notebook uses the challenge word generated by Saysafe and a made-up passing voice score to simulate an approved reply.

In [ ]:
approvals = ApprovalGuard(
    secret=secrets.token_bytes(32),
    policy={"rules": [{"id": "notebook-demo", "when": {}, "tier": "voice_challenge", "reason": "Notebook demo"}]},
)
transfer = Action(
    type='send_money',
    counterparty='Maya',
    amount=Decimal('25'),
    destination='@maya',
)

step = approvals.start(transfer, room=Room.ALONE)
print('Agent asks:', step.say)
print('Status:', step.status)

# Simulated input: this is not a real microphone or speaker check.
step = approvals.reply(step, step.challenge_word, voice_score=0.99)
print('After simulated confirmation:', step.status)

# A host app calls verify immediately before it runs the real tool.
approvals.verify(step.token, transfer)
print('Exact approved action verified; no transfer was executed.')

Expected: the status changes from `awaiting_reply` to `approved`, and the exact action verifies. The token is only for this notebook run; re-running the cell creates a new key and a new token.

## 3. Check that changing the transfer is refused

An approval for $25 must not authorize $26. This cell makes a fresh approval, changes the amount, and prints the refusal.

In [ ]:
fresh = ApprovalGuard(
    secret=secrets.token_bytes(32),
    policy={"rules": [{"id": "notebook-demo", "when": {}, "tier": "voice_challenge", "reason": "Notebook demo"}]},
)
pending = fresh.start(transfer, room=Room.ALONE)
approved = fresh.reply(pending, pending.challenge_word, voice_score=0.99)
changed_transfer = transfer.model_copy(update={"amount": Decimal('26')})

try:
    fresh.verify(approved.token, changed_transfer)
except Refused as error:
    print('Refused changed action:', error.reason)

Expected: `Refused changed action: action_changed`. This is the check the host app must keep immediately before executing a consequential tool call.